# Gestión de Tareas

**Menú del programa:**
1. Agregar nueva tarea
2. Agregar contenidos a la tarea
3. Mostrar estado de cada tarea
4. Eliminar tarea
5. Marcar contenido como realizado/pendiente o eliminarlo
6. Materias (crear)
7. Agregar tarea a materias
8. Salir del programa

> **Nota:** este código está pensado para ejecutarse en consola (usa `input()`), por lo que dentro del notebook las celdas de código se muestran con fines explicativos. Si querés ejecutarlo de forma interactiva, podés correr la última celda dentro de un entorno que soporte `input()` (Jupyter clásico o consola).

In [ ]:
from collections import deque

## 1. Clase `ElementoBase`

Es la clase **padre** (base) de la que heredan `Tarea` y `Materia`. Su único propósito es guardar el atributo común `nombre`, evitando repetir código (principio de herencia / reutilización).

In [ ]:
class ElementoBase:
    def __init__(self, nombre):
        self.nombre = nombre

## 2. Clase `Tarea`

Hereda de `ElementoBase`. Cada tarea tiene:
- `nombre`: heredado de la clase base.
- `contenidos`: una **pila** (`deque`) donde cada elemento es un diccionario `{"texto": ..., "estado": ...}`.

### Método `mostrar_detalle()`
Imprime el nombre de la tarea y, si tiene contenidos, los recorre con `reversed()` para respetar el orden de pila (el contenido agregado más recientemente se muestra primero, numerado desde 1).

**Validación:** si `self.contenidos` está vacío (evaluación `if self.contenidos:` es `False` en una deque vacía), se imprime `"(Sin contenidos)"` en lugar de intentar iterar sobre una colección vacía.

In [ ]:
class Tarea(ElementoBase):
    def __init__(self, nombre):
        super().__init__(nombre)
        self.contenidos = deque()

    def mostrar_detalle(self):
        print(f"    - Tarea: {self.nombre}")
        if self.contenidos:
            for i, c in enumerate(reversed(self.contenidos), 1):
                print(f"      {i}. {c['texto']} ({c['estado']})")
        else:
            print("      (Sin contenidos)")

## 3. Clase `Materia`

También hereda de `ElementoBase`. Agrega el atributo `tareas`, que es otra **pila** (`deque`) de objetos `Tarea`. Sirve para agrupar tareas bajo una materia académica.

In [ ]:
class Materia(ElementoBase):
    def __init__(self, nombre):
        super().__init__(nombre)
        self.tareas = deque()

## 4. Clase `GestorTareas`

Es la clase principal que controla todo el programa: el menú, la entrada del usuario y la lógica de negocio.

### Atributos
- `materias`: lista de objetos `Materia`.
- `tareas_libres`: pila (`deque`) de tareas que **todavía no** han sido asignadas a ninguna materia.

### Método `ejecutar()`
Contiene el **bucle principal** del programa (`while True`). En cada vuelta:
1. Muestra el menú.
2. Pide al usuario un número de opción.
3. Ejecuta el método correspondiente según la opción elegida.

**Validaciones:**
- `try/except ValueError`: si el usuario escribe algo que no es un número entero, se captura el error y se muestra un mensaje claro en lugar de que el programa se rompa.
- `except Exception as e`: captura genérica para cualquier otro error inesperado, mostrando el detalle sin detener la ejecución.
- `else: print("Error: Opción fuera de rango (1-8).")`: valida que la opción numérica esté dentro del rango permitido del menú.
- La opción `8` es la única que rompe el bucle (`break`), terminando el programa.

In [ ]:
class GestorTareas:
    def __init__(self):
        self.materias = []
        self.tareas_libres = deque()

    def ejecutar(self):
        while True:
            self.mostrar_menu()
            try:
                opcion = int(input("\nSeleccione una opción (1-8): "))
                print("-" * 40)

                if opcion == 1:
                    self.agregar_tarea()
                elif opcion == 2:
                    self.agregar_contenido()
                elif opcion == 3:
                    self.mostrar_estado()
                elif opcion == 4:
                    self.eliminar_tarea()
                elif opcion == 5:
                    self.gestionar_estado_contenido()
                elif opcion == 6:
                    self.crear_materia()
                elif opcion == 7:
                    self.asignar_tarea_materia()
                elif opcion == 8:
                    print("Saliendo del programa. ¡Hasta luego!")
                    break
                else:
                    print("Error: Opción fuera de rango (1-8).")

            except ValueError:
                print("Error de validación: Debe ingresar un número entero válido.")
            except Exception as e:
                print(f"Ocurrió un error inesperado: {e}")

### Método `mostrar_menu()`
Solo imprime las opciones disponibles en pantalla. No contiene lógica ni validaciones.

In [ ]:
    def mostrar_menu(self):
        print("\n" + "=" * 40)
        print("    GESTIÓN DE TAREAS (PILAS CON DEQUE)")
        print("=" * 40)
        print("1. Agregar nueva tarea")
        print("2. Agregar contenidos a la tarea")
        print("3. Mostrar estado de cada tarea (Avances)")
        print("4. Eliminar tarea")
        print("5. Marcar contenido como realizado/pendiente o eliminarlo")
        print("6. Materias (Crear)")
        print("7. Agregar tarea a materias")
        print("8. Salir del programa")

### Método `agregar_tarea()`
Pide nombres de tareas en un bucle `while True` y las va **apilando** (`append`) en `tareas_libres`.

**Validaciones:**
- `if nombre == '0': break` — permite salir del submenú escribiendo `0`.
- `if not nombre: ... continue` — evita crear tareas con nombre vacío (usa `.strip()` para descartar espacios en blanco).

In [ ]:
    def agregar_tarea(self):
        while True:
            nombre = input("Ingrese el nombre de la tarea (o '0' para volver): ").strip()
            if nombre == '0':
                break
            if not nombre:
                print("El nombre no puede estar vacío.")
                continue
            self.tareas_libres.append(Tarea(nombre))
            print(f"¡Tarea '{nombre}' apilada con éxito!")

### Método `agregar_contenido()`
Primero llama a `seleccionar_tarea()` (ver más abajo) para elegir sobre qué tarea trabajar. Si no se selecciona ninguna (`None`), termina con `return`.

Luego entra en un bucle para ir agregando contenidos, cada uno guardado como diccionario `{"texto": texto, "estado": "pendiente"}` — todo contenido nuevo nace con estado `"pendiente"`.

**Validaciones:** iguales al método anterior — `'0'` para salir y texto vacío rechazado con `.strip()`.

In [ ]:
    def agregar_contenido(self):
        tarea = self.seleccionar_tarea()
        if not tarea:
            return

        while True:
            texto = input(f"Escriba el contenido para '{tarea.nombre}' (o '0' para volver): ").strip()
            if texto == '0':
                break
            if not texto:
                print("El contenido no puede estar vacío.")
                continue
            tarea.contenidos.append({"texto": texto, "estado": "pendiente"})
            print("Contenido agregado a la Pila.")

### Método `mostrar_estado()`
Muestra un resumen general: primero las `tareas_libres` (recorridas con `reversed()` para respetar el orden de pila), y luego, para cada materia, sus tareas asignadas (también con `reversed()`).

**Validación:** si no hay ni tareas libres ni materias registradas, se informa al usuario y se corta la ejecución del método con `return` en lugar de mostrar un estado vacío sin sentido.

In [ ]:
    def mostrar_estado(self):
        if not self.tareas_libres and not self.materias:
            print("No hay tareas ni materias registradas.")
            return

        print("\n--- ESTADO DE AVANCES (ESTRUCTURA DE PILAS) ---")
        if self.tareas_libres:
            print("Tareas Generales (Pila):")
            for tarea in reversed(self.tareas_libres):
                tarea.mostrar_detalle()

        for materia in self.materias:
            print(f"\nMateria: {materia.nombre} (Pila de tareas)")
            if not materia.tareas:
                print("    (Sin tareas asignadas)")
            for tarea in reversed(materia.tareas):
                tarea.mostrar_detalle()

### Método `eliminar_tarea()`
Selecciona una tarea con `seleccionar_tarea()` y la elimina de donde esté: de `tareas_libres` si está ahí, o recorriendo todas las materias para quitarla de la que la contenga.

**Validación:** si `seleccionar_tarea()` devuelve `None` (usuario canceló o no hay tareas), el método termina con `return` sin intentar eliminar nada.

In [ ]:
    def eliminar_tarea(self):
        tarea = self.seleccionar_tarea()
        if not tarea:
            return

        if tarea in self.tareas_libres:
            self.tareas_libres.remove(tarea)
        for materia in self.materias:
            if tarea in materia.tareas:
                materia.tareas.remove(tarea)
        print(f"Tarea '{tarea.nombre}' eliminada correctamente.")

### Método `gestionar_estado_contenido()`
Permite modificar o eliminar un contenido específico de una tarea:
1. Selecciona la tarea.
2. Muestra sus contenidos numerados.
3. Pide el número del contenido a modificar.
4. Ofrece 3 sub-opciones: marcar como `realizado`, marcar como `pendiente`, o eliminar por completo.

**Validaciones:**
- `if not tarea or not tarea.contenidos:` — corta si no hay tarea seleccionada o si la tarea no tiene contenidos.
- `if 0 <= idx < len(lista_contenidos):` — valida que el índice ingresado esté dentro del rango de contenidos existentes (recordando que el usuario ingresa un número basado en 1, por eso se resta 1 antes).
- `else: print("Opción no válida.")` dentro del sub-menú — valida que la opción `1`, `2` o `3` sea correcta.
- `else: print("Número de contenido inválido.")` — mensaje para índices fuera de rango.

Nota técnica: como `deque` no soporta indexado eficiente ni modificación directa por posición, el código convierte temporalmente el `deque` a `list` (`lista_contenidos = list(tarea.contenidos)`), hace el cambio, y luego reconstruye el `deque` (`tarea.contenidos = deque(lista_contenidos)`).

In [ ]:
    def gestionar_estado_contenido(self):
        tarea = self.seleccionar_tarea()
        if not tarea or not tarea.contenidos:
            print("La tarea seleccionada no tiene contenidos.")
            return

        tarea.mostrar_detalle()
        lista_contenidos = list(tarea.contenidos)
        idx = int(input("Ingrese el número del contenido que desea modificar/eliminar: ")) - 1

        if 0 <= idx < len(lista_contenidos):
            print("\n¿Qué desea hacer con este contenido?")
            print("1. Cambiar estado a (realizado)")
            print("2. Cambiar estado a (pendiente)")
            print("3. Eliminarlo por completo")
            op = input("Seleccione una opción: ").strip()

            if op == '1':
                lista_contenidos[idx]["estado"] = "realizado"
                tarea.contenidos = deque(lista_contenidos)
                print("¡Contenido marcado como (realizado)!")
            elif op == '2':
                lista_contenidos[idx]["estado"] = "pendiente"
                tarea.contenidos = deque(lista_contenidos)
                print("¡Contenido marcado como (pendiente)!")
            elif op == '3':
                eliminado = lista_contenidos.pop(idx)
                tarea.contenidos = deque(lista_contenidos)
                print(f"Contenido '{eliminado['texto']}' eliminado por completo.")
            else:
                print("Opción no válida.")
        else:
            print("Número de contenido inválido.")

### Método `crear_materia()`
Crea una nueva `Materia` y la agrega a la lista `self.materias`.

**Validación:** `if not nombre:` — rechaza nombres vacíos (tras aplicar `.strip()`).

In [ ]:
    def crear_materia(self):
        nombre = input("Ingrese el nombre de la nueva materia: ").strip()
        if not nombre:
            print("El nombre no puede estar vacío.")
            return
        self.materias.append(Materia(nombre))
        print(f"Materia '{nombre}' creada con éxito.")

### Método `asignar_tarea_materia()`
Mueve una tarea desde `tareas_libres` (o desde otra materia) hacia la pila de tareas de una materia elegida.

**Validaciones:**
- `if not self.materias:` — no permite continuar si no existe ninguna materia creada.
- `if not tarea: return` — si no se seleccionó una tarea válida.
- `if 0 <= idx < len(self.materias):` — valida que el número de materia ingresado exista; si no, muestra `"Número de materia inválido."`.

In [ ]:
    def asignar_tarea_materia(self):
        if not self.materias:
            print("Debe crear al menos una materia primero.")
            return

        tarea = self.seleccionar_tarea()
        if not tarea:
            return

        print("\nSeleccione la materia destino:")
        for i, mat in enumerate(self.materias, 1):
            print(f"{i}. {mat.nombre}")

        idx = int(input("Número de materia: ")) - 1
        if 0 <= idx < len(self.materias):
            self.materias[idx].tareas.append(tarea)
            if tarea in self.tareas_libres:
                self.tareas_libres.remove(tarea)
            print(f"Tarea asignada a la Pila de la materia '{self.materias[idx].nombre}'.")
        else:
            print("Número de materia inválido.")

### Método `seleccionar_tarea()`
Método auxiliar reutilizado por varios otros métodos. Junta en una sola lista `todas` las tareas libres más las tareas de todas las materias, las muestra numeradas y pide al usuario que elija una.

**Validaciones:**
- `if not todas:` — informa si no hay ninguna tarea en el sistema.
- `if idx == -1: return None` — permite cancelar escribiendo `0`.
- `if 0 <= idx < len(todas): return todas[idx]` — valida que el número elegido exista.
- Si el índice está fuera de rango, imprime `"Número de tarea fuera de rango."` y devuelve `None`.

**Detalle a tener en cuenta:** la línea `t.name if hasattr(t, 'name') else t.nombre` nunca usará `t.name`, porque la clase `ElementoBase` solo define el atributo `nombre` (no `name`). En la práctica, siempre se ejecuta la rama `t.nombre`; ese `hasattr` es código defensivo que no se activa con las clases actuales.

In [ ]:
    def seleccionar_tarea(self):
        todas = list(self.tareas_libres) + [t for m in self.materias for t in m.tareas]
        if not todas:
            print("No hay tareas registradas en el sistema.")
            return None

        print("\nListado de Tareas:")
        for i, t in enumerate(todas, 1):
            print(f"{i}. {t.name if hasattr(t, 'name') else t.nombre}")

        idx = int(input("Seleccione el número de la tarea (o '0' para cancelar): ")) - 1
        if idx == -1:
            return None
        if 0 <= idx < len(todas):
            return todas[idx]

        print("Número de tarea fuera de rango.")
        return None

## 5. Punto de entrada del programa

Al ejecutar el archivo directamente (`python archivo.py`), se crea una instancia de `GestorTareas` y se llama a `ejecutar()`, arrancando el bucle del menú.

In [ ]:
if __name__ == "__main__":
    app = GestorTareas()
    app.ejecutar()

## 6. Resumen de validaciones del programa

| Método | Validación | Qué evita |
|---|---|---|
| `ejecutar` | `try/except ValueError` | Que el programa se caiga si el usuario no ingresa un número |
| `ejecutar` | `except Exception` | Que un error inesperado detenga el programa |
| `ejecutar` | rango 1-8 | Opciones de menú inválidas |
| `agregar_tarea` / `agregar_contenido` / `crear_materia` | `.strip()` + `if not nombre/texto` | Nombres o contenidos vacíos |
| `agregar_tarea` / `agregar_contenido` | `'0'` para volver | Bucle infinito sin salida |
| `mostrar_estado` | `if not tareas_libres and not materias` | Mostrar un estado vacío sin sentido |
| `eliminar_tarea` / `agregar_contenido` / `asignar_tarea_materia` | `if not tarea` (de `seleccionar_tarea`) | Operar sobre una tarea inexistente |
| `gestionar_estado_contenido` | `if not tarea or not tarea.contenidos` | Modificar contenidos que no existen |
| `gestionar_estado_contenido` | `0 <= idx < len(lista_contenidos)` | Índices de contenido fuera de rango |
| `gestionar_estado_contenido` | validación de sub-opción (`1`/`2`/`3`) | Operación inválida sobre el contenido |
| `asignar_tarea_materia` | `if not self.materias` | Asignar tareas sin materias creadas |
| `asignar_tarea_materia` | `0 <= idx < len(self.materias)` | Índice de materia fuera de rango |
| `seleccionar_tarea` | `if not todas` | Buscar tareas cuando no existe ninguna |
| `seleccionar_tarea` | `idx == -1` | Cancelar la selección |
| `seleccionar_tarea` | `0 <= idx < len(todas)` | Índice de tarea fuera de rango |

### Sobre el uso de `deque` como pila
El programa usa `deque` en lugar de `list` para representar pilas porque `deque` está optimizado para agregar y quitar elementos por los extremos (`append`/`appendleft`, `pop`/`popleft`) en tiempo O(1). Aunque en este código no se usa `pop()` directamente sobre las deques (las eliminaciones se hacen con `.remove()` buscando el elemento), el orden LIFO se simula visualmente mostrando siempre los elementos con `reversed()`, de modo que lo último agregado aparece primero en pantalla — tal como se vería el "tope" de una pila.